In [0]:
# Cell 1: load the daily series
import numpy as np
import pandas as pd

sales = spark.table("ecom_dev.reporting.sales_daily").select("full_date", "net_sales").toPandas()
sales["full_date"] = pd.to_datetime(sales["full_date"])
s = sales.set_index("full_date")["net_sales"].astype(float).sort_index()
print(len(s), s.index.min().date(), s.index.max().date())

In [0]:
# Cell 2: the three baselines. Each sees only the history before the forecast start.
def moving_avg_28(h, dates):
    return np.full(len(dates), h.iloc[-28:].mean())

def naive_weekly(h, dates):
    last = h.iloc[-28:]
    by_dow = last.groupby(last.index.dayofweek).mean()
    return np.array([by_dow[d.dayofweek] for d in dates])

def seasonal_yoy(h, dates):
    end = h.index.max()
    recent = h.loc[end - pd.Timedelta(days=27): end].sum()
    year_ago = h.loc[end - pd.Timedelta(days=27 + 364): end - pd.Timedelta(days=364)].sum()
    growth = recent / year_ago
    return np.array([h.get(d - pd.Timedelta(days=364), np.nan) * growth for d in dates])

In [0]:
# Cell 3: rolling-origin backtest, then the scoreboard
def backtest(fn, horizons=(7, 30, 90)):
    rows = []
    for origin in pd.date_range("2025-12-01", "2026-10-01", freq="7D"):
        history = s[s.index < origin]
        for horizon in horizons:
            dates = pd.date_range(origin, periods=horizon)
            if dates[-1] > s.index.max():
                continue                      # not enough actuals to score this window
            forecast = fn(history, dates)
            actual = s.reindex(dates).values
            rows.append((fn.__name__, origin, horizon,
                         np.abs(actual - forecast).sum(), actual.sum(), (forecast - actual).sum()))
    return pd.DataFrame(rows, columns=["model", "origin", "horizon", "abs_err", "actual", "bias"])

results = pd.concat([backtest(f) for f in (moving_avg_28, naive_weekly, seasonal_yoy)])
score = (results.groupby(["model", "horizon"])
         .agg(abs_err=("abs_err", "sum"), actual=("actual", "sum"), bias=("bias", "sum"),
              windows=("origin", "count")).reset_index())
score["WAPE_pct"] = (100 * score["abs_err"] / score["actual"]).round(1)
score["bias_pct"] = (100 * score["bias"] / score["actual"]).round(1)
display(score[["model", "horizon", "windows", "WAPE_pct", "bias_pct"]].sort_values(["horizon", "WAPE_pct"]))

In [0]:
%pip install prophet

In [0]:
# Cell 4: the Prophet forecast and its backtest
import logging
from prophet import Prophet

logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
logging.getLogger("prophet").setLevel(logging.WARNING)


def prophet_forecast(history, horizon):
    """Fit on the history only, then predict the next `horizon` days: (forecast, low, high)."""
    m = Prophet(growth="linear", seasonality_mode="multiplicative",
                yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False,
                interval_width=0.8)
    m.fit(pd.DataFrame({"ds": history.index, "y": history.values}))
    out = m.predict(m.make_future_dataframe(periods=horizon, include_history=False))
    return out["yhat"].values, out["yhat_lower"].values, out["yhat_upper"].values


def backtest_prophet(origins, horizons=(7, 30, 90)):
    rows = []
    for origin in origins:
        history = s[s.index < origin]
        yhat, _, _ = prophet_forecast(history, max(horizons))   # one fit serves all three horizons
        for horizon in horizons:
            dates = pd.date_range(origin, periods=horizon)
            if dates[-1] > s.index.max():
                continue
            actual = s.reindex(dates).values
            forecast = yhat[:horizon]
            rows.append(("prophet", origin, horizon,
                         np.abs(actual - forecast).sum(), actual.sum(), (forecast - actual).sum()))
    return pd.DataFrame(rows, columns=["model", "origin", "horizon", "abs_err", "actual", "bias"])

In [0]:
# Cell 5: all models on the same windows, one scoreboard
origins = pd.date_range("2025-12-01", "2026-10-01", freq="14D")
prophet_res = backtest_prophet(origins)

all_res = pd.concat([results[results["origin"].isin(origins)], prophet_res])
board = (all_res.groupby(["model", "horizon"])
         .agg(abs_err=("abs_err", "sum"), actual=("actual", "sum"), bias=("bias", "sum"),
              windows=("origin", "count")).reset_index())
board["WAPE_pct"] = (100 * board["abs_err"] / board["actual"]).round(1)
board["bias_pct"] = (100 * board["bias"] / board["actual"]).round(1)
display(board[["model", "horizon", "windows", "WAPE_pct", "bias_pct"]].sort_values(["horizon", "WAPE_pct"]))

In [0]:
# Cell 6: the business event calendar, saved as a table so training and scoring share one source
events = [
    # (event_name, start, end, status)
    ("black_friday",      "2024-11-25", "2024-12-02", "actual"),
    ("black_friday",      "2025-11-26", "2025-11-30", "actual"),
    ("black_friday",      "2026-11-25", "2026-11-29", "planned"),
    ("christmas",         "2024-12-24", "2024-12-26", "actual"),
    ("christmas",         "2025-12-24", "2025-12-26", "actual"),
    ("christmas",         "2026-12-24", "2026-12-26", "planned"),
    ("republic_day_sale", "2025-01-24", "2025-01-26", "actual"),
    ("republic_day_sale", "2026-01-24", "2026-01-26", "actual"),
    ("republic_day_sale", "2027-01-24", "2027-01-26", "planned"),
    ("valentines_sale",   "2025-02-13", "2025-02-14", "actual"),
    ("valentines_sale",   "2026-02-13", "2026-02-14", "actual"),
    ("valentines_sale",   "2027-02-13", "2027-02-14", "planned"),
    ("midyear_sale",      "2025-06-20", "2025-06-22", "actual"),
    ("midyear_sale",      "2026-06-20", "2026-06-22", "actual"),
    ("aug_flash_sale",    "2025-08-08", "2025-08-10", "actual"),
    ("independence_sale", "2025-08-13", "2025-08-15", "actual"),
    ("independence_sale", "2026-08-13", "2026-08-15", "actual"),
    ("festive_kickoff",   "2025-09-22", "2025-10-02", "actual"),
    ("festive_kickoff",   "2026-09-21", "2026-10-01", "actual"),
    ("festive_main",      "2025-10-12", "2025-10-22", "actual"),
    ("festive_main",      "2026-10-10", "2026-10-31", "actual"),
]
(spark.createDataFrame(events, "event_name string, start_date string, end_date string, status string")
      .selectExpr("event_name", "to_date(start_date) AS start_date", "to_date(end_date) AS end_date", "status")
      .write.mode("overwrite").saveAsTable("ecom_dev.ml.event_calendar"))

In [0]:
# Cell 7: Prophet again, now told when the events happen
cal = spark.table("ecom_dev.ml.event_calendar").toPandas()
HOLIDAYS = pd.DataFrame([{"holiday": r.event_name, "ds": d}
                         for r in cal.itertuples()
                         for d in pd.date_range(r.start_date, r.end_date)])


def prophet_forecast(history, horizon):          # replaces the Cell 4 version; backtest_prophet picks it up
    m = Prophet(growth="linear", seasonality_mode="multiplicative",
                yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False,
                interval_width=0.8, holidays=HOLIDAYS)
    m.fit(pd.DataFrame({"ds": history.index, "y": history.values}))
    out = m.predict(m.make_future_dataframe(periods=horizon, include_history=False))
    return out["yhat"].values, out["yhat_lower"].values, out["yhat_upper"].values


prophet_ev = backtest_prophet(origins)
prophet_ev["model"] = "prophet_events"

In [0]:
# Cell 8: scoreboard with a second view: the error on the window TOTAL (what a weekly/monthly chart shows)
all_res = pd.concat([results[results["origin"].isin(origins)], prophet_res, prophet_ev])
all_res["abs_window_err"] = all_res["bias"].abs()          # |forecast total - actual total| per window
board = (all_res.groupby(["model", "horizon"])
         .agg(abs_err=("abs_err", "sum"), abs_window_err=("abs_window_err", "sum"),
              actual=("actual", "sum"), bias=("bias", "sum"), windows=("origin", "count")).reset_index())
board["daily_WAPE_pct"]  = (100 * board["abs_err"] / board["actual"]).round(1)
board["window_WAPE_pct"] = (100 * board["abs_window_err"] / board["actual"]).round(1)
board["bias_pct"]        = (100 * board["bias"] / board["actual"]).round(1)
display(board[["model", "horizon", "windows", "daily_WAPE_pct", "window_WAPE_pct", "bias_pct"]]
        .sort_values(["horizon", "daily_WAPE_pct"]))

In [0]:
# Cell 9: same backtest, with the strength of the event effects capped
HPS = 10.0


def prophet_forecast(history, horizon):          # replaces the Cell 7 version
    m = Prophet(growth="linear", seasonality_mode="multiplicative",
                yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False,
                interval_width=0.8, holidays=HOLIDAYS, holidays_prior_scale=HPS)
    m.fit(pd.DataFrame({"ds": history.index, "y": history.values}))
    out = m.predict(m.make_future_dataframe(periods=horizon, include_history=False))
    return out["yhat"].values, out["yhat_lower"].values, out["yhat_upper"].values


variants = []
for HPS in (1.0, 3.0):                            # each run is ~22 fits
    res = backtest_prophet(origins)
    res["model"] = f"prophet_events_hps{HPS:g}"
    variants.append(res)

In [0]:
# Cell 10: scoreboard (plain Prophet and the best baseline kept for reference)
def scoreboard(frames):
    df = pd.concat(frames)
    df["abs_window_err"] = df["bias"].abs()
    b = (df.groupby(["model", "horizon"])
         .agg(abs_err=("abs_err", "sum"), abs_window_err=("abs_window_err", "sum"),
              actual=("actual", "sum"), bias=("bias", "sum"), windows=("origin", "count")).reset_index())
    b["daily_WAPE_pct"]  = (100 * b["abs_err"] / b["actual"]).round(1)
    b["window_WAPE_pct"] = (100 * b["abs_window_err"] / b["actual"]).round(1)
    b["bias_pct"]        = (100 * b["bias"] / b["actual"]).round(1)
    return b[["model", "horizon", "windows", "daily_WAPE_pct", "window_WAPE_pct", "bias_pct"]]


base = results[(results["origin"].isin(origins)) & (results["model"] == "seasonal_yoy")]
display(scoreboard([base, prophet_res, prophet_ev] + variants).sort_values(["horizon", "daily_WAPE_pct"]))

In [0]:
# Cell 11: how often does the 80% interval actually contain the real value?
HPS = 1.0            # the chosen setting (the loop in Cell 9 left it at 3.0)


def coverage_backtest(origins, horizons=(7, 30, 90)):
    rows = []
    for origin in origins:
        history = s[s.index < origin]
        _, low, high = prophet_forecast(history, max(horizons))
        for horizon in horizons:
            dates = pd.date_range(origin, periods=horizon)
            if dates[-1] > s.index.max():
                continue
            actual = s.reindex(dates).values
            inside = (actual >= low[:horizon]) & (actual <= high[:horizon])
            rows.append((origin, horizon, int(inside.sum()), horizon))
    return pd.DataFrame(rows, columns=["origin", "horizon", "inside", "days"])


cov = coverage_backtest(origins)
cover = cov.groupby("horizon").agg(inside=("inside", "sum"), days=("days", "sum")).reset_index()
cover["coverage_pct"] = (100 * cover["inside"] / cover["days"]).round(1)
display(cover)

In [0]:
# Cell 12: calibrate the intervals from backtest errors, and test the idea on later windows
def daily_backtest(origins, max_h=90):
    frames = []
    for origin in origins:
        history = s[s.index < origin]
        yhat, _, _ = prophet_forecast(history, max_h)
        dates = pd.date_range(origin, periods=max_h)
        keep = dates <= s.index.max()
        frames.append(pd.DataFrame({"origin": origin, "k": np.arange(1, max_h + 1)[keep], "ds": dates[keep],
                                    "actual": s.reindex(dates[keep]).values, "yhat": yhat[keep]}))
    d = pd.concat(frames)
    d["ratio"] = d["actual"] / d["yhat"]
    d["bucket"] = pd.cut(d["k"], [0, 7, 30, 90], labels=["1-7", "8-30", "31-90"])
    return d


bt = daily_backtest(origins)

cut = origins[len(origins) // 2]
train = bt[(bt["origin"] < cut) & (bt["ds"] < cut)]      # only errors known before the cut date
test = bt[bt["origin"] >= cut]                           # later windows the calibration never saw

q = (train.groupby("bucket", observed=True)["ratio"].quantile([0.1, 0.5, 0.9]).unstack())
q.columns = ["q10", "q50", "q90"]
display(q.reset_index())

test = test.join(q, on="bucket")
test["inside"] = ((test["actual"] >= test["yhat"] * test["q10"])
                  & (test["actual"] <= test["yhat"] * test["q90"]))
display(test.groupby("bucket", observed=True)["inside"].agg(["mean", "count"]).reset_index()
        .assign(coverage_pct=lambda x: (100 * x["mean"]).round(1))[["bucket", "count", "coverage_pct"]])

In [0]:
# Cell 13: final calibration on all backtest errors, saved as a table
def block_ratio(bt, size):
    """actual / forecast TOTALS over consecutive `size`-day blocks of each forecast window (full blocks only)."""
    b = bt.copy()
    b["block"] = (b["k"] - 1) // size
    g = (b.groupby(["origin", "block"])
          .agg(days=("k", "count"), actual=("actual", "sum"), yhat=("yhat", "sum")).reset_index())
    g = g[g["days"] == size]
    g["ratio"] = g["actual"] / g["yhat"]
    return g


def q3(series):
    return series.quantile([0.1, 0.5, 0.9]).tolist()


rows = []
for bucket, grp in bt.groupby("bucket", observed=True):          # days: three horizon buckets
    rows.append(("day", str(bucket), *q3(grp["ratio"]), len(grp)))
wk = block_ratio(bt, 7)
rows.append(("week", "all", *q3(wk["ratio"]), len(wk)))           # weeks: all 7-day blocks pooled
mo = block_ratio(bt, 30)
rows.append(("month", "all", *q3(mo["ratio"]), len(mo)))          # months: all 30-day blocks pooled

calib = pd.DataFrame(rows, columns=["grain", "bucket", "q10", "q50", "q90", "n"])
calib["model_name"] = "prophet_events_hps1"
calib["calibrated_at"] = pd.Timestamp.now()
display(calib)

(spark.createDataFrame(calib.astype({"n": "int"}))
      .write.mode("overwrite").saveAsTable("ecom_dev.ml.forecast_calibration"))

In [0]:
# Cell 14: train on all data, log to MLflow, register in Unity Catalog
import mlflow
import mlflow.prophet
from mlflow import MlflowClient
from mlflow.models import infer_signature

mlflow.set_registry_uri("databricks-uc")
user = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{user}/ecom_sales_forecast")

MODEL_NAME = "ecom_dev.ml.sales_forecast_prophet"
HPS = 1.0
chosen = scoreboard([variants[0]])          # backtest numbers of the chosen setting (holidays_prior_scale = 1)

with mlflow.start_run(run_name="prophet_events_final") as run:
    m = Prophet(growth="linear", seasonality_mode="multiplicative",
                yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False,
                interval_width=0.8, holidays=HOLIDAYS, holidays_prior_scale=HPS)
    m.fit(pd.DataFrame({"ds": s.index, "y": s.values}))

    future = m.make_future_dataframe(periods=90, include_history=False)
    sample_out = m.predict(future)

    mlflow.log_params({
        "growth": "linear", "seasonality_mode": "multiplicative", "holidays_prior_scale": HPS,
        "interval_width": 0.8, "train_start": str(s.index.min().date()), "train_end": str(s.index.max().date()),
        "train_days": len(s), "event_rows": len(HOLIDAYS), "series": "total_net_sales_daily",
    })
    for _, r in chosen.iterrows():
        h = int(r["horizon"])
        mlflow.log_metric(f"backtest_daily_wape_h{h}", float(r["daily_WAPE_pct"]))
        mlflow.log_metric(f"backtest_window_wape_h{h}", float(r["window_WAPE_pct"]))
        mlflow.log_metric(f"backtest_bias_h{h}", float(r["bias_pct"]))
    mlflow.log_table(calib.assign(calibrated_at=calib["calibrated_at"].astype(str)), "calibration.json")

    mlflow.prophet.log_model(m, "model", signature=infer_signature(future, sample_out),
                             registered_model_name=MODEL_NAME)

client = MlflowClient()
version = max(int(v.version) for v in client.search_model_versions(f"name='{MODEL_NAME}'"))
client.set_registered_model_alias(MODEL_NAME, "champion", version)
print(f"run_id={run.info.run_id}  model={MODEL_NAME}  version={version}  alias=champion")

In [0]:
# Cell 15: what is dragging the forecast down: trend, yearly seasonality or events?
import mlflow
import mlflow.prophet

mlflow.set_registry_uri("databricks-uc")
m = mlflow.prophet.load_model("models:/ecom_dev.ml.sales_forecast_prophet@champion")

full = m.predict(m.make_future_dataframe(periods=90, include_history=True))
full["month"] = full["ds"].dt.to_period("M").astype(str)
parts = (full.groupby("month")
         .agg(trend=("trend", "mean"), yearly=("yearly", "mean"),
              events=("holidays", "mean"), weekly=("weekly", "mean"), yhat=("yhat", "mean"))
         .round(3).reset_index())
display(parts)

In [0]:
# Cell 16: is the flat trend real, or caused by how the festive events are modelled? (2 fits)
import logging
import numpy as np
import pandas as pd
from prophet import Prophet

logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
logging.getLogger("prophet").setLevel(logging.WARNING)

sales = spark.table("ecom_dev.reporting.sales_daily").select("full_date", "net_sales").toPandas()
sales["full_date"] = pd.to_datetime(sales["full_date"])
s = sales.set_index("full_date")["net_sales"].astype(float).sort_index()
cal = spark.table("ecom_dev.ml.event_calendar").toPandas()
q50_month = float(spark.table("ecom_dev.ml.forecast_calibration").where("grain = 'month'").first()["q50"])


def holidays_from(cal, split=()):
    c = cal.copy()
    for name in split:                       # give each year's occurrence of this event its own effect
        m = c["event_name"] == name
        c.loc[m, "event_name"] = name + "_" + c.loc[m, "start_date"].astype(str).str[:4]
    return pd.DataFrame([{"holiday": r.event_name, "ds": d}
                         for r in c.itertuples() for d in pd.date_range(r.start_date, r.end_date)])


def fit(history, holidays):
    m = Prophet(growth="linear", seasonality_mode="multiplicative", yearly_seasonality=True,
                weekly_seasonality=True, daily_seasonality=False, interval_width=0.8,
                holidays=holidays, holidays_prior_scale=1.0)
    m.fit(pd.DataFrame({"ds": history.index, "y": history.values}))
    return m


def with_months(m, periods):
    full = m.predict(m.make_future_dataframe(periods=periods, include_history=True))
    full["month"] = full["ds"].dt.to_period("M").astype(str)
    return full


# (1) current setup, but only data up to 31 Aug 2026 (before the festive season)
f1 = with_months(fit(s[s.index <= "2026-08-31"], holidays_from(cal)), 0)
t1 = f1[f1["month"] >= "2026-03"].groupby("month")["trend"].mean().round(0)

# (2) all data, with the two festive_main windows (11 vs 22 days) given their own effects
f2 = with_months(fit(s, holidays_from(cal, split=("festive_main",))), 90)
t2 = f2[f2["month"] >= "2026-03"].groupby("month")["trend"].mean().round(0)

display(pd.DataFrame({"trend_data_to_aug": t1, "trend_all_data_split_events": t2}).reset_index())

future = f2[f2["ds"] > s.index.max()].reset_index(drop=True)
blocks = [round(future["yhat"].iloc[i:i + 30].sum() * q50_month) for i in (0, 30, 60)]
print("Split-events model, 30-day blocks (calibrated):", blocks)
print("Current champion:                              [12374840, 8881755, 7770447]")